# 03 — IndoBERT Fine-tuning: Sentiment Analysis (3-class)

Fine-tuning `indobenchmark/indobert-base-p1` on Tokopedia iPhone 17 reviews.

**Labels:** Positif (0) · Negatif (1) · Netral (2)  
**Dataset:** `data/processed/tokopedia_reviews_clean.csv` (1556 rows)  
**Distribution:** Positif≈1495, Negatif≈49, Netral≈12 → class weights to handle imbalance

In [ ]:
## 0. Google Colab Setup  (skip if running locally)
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Google Colab detected — mounting Drive & installing packages …")

    from google.colab import drive
    drive.mount("/content/drive")

    # ── Adjust DRIVE_PROJECT to match your Drive folder path ──────────
    DRIVE_PROJECT = "/content/drive/MyDrive/xai_lime_vs_shap"

    if os.path.isdir(DRIVE_PROJECT):
        os.chdir(DRIVE_PROJECT)
        print(f"CWD → {DRIVE_PROJECT}")
    else:
        print(f"WARNING: '{DRIVE_PROJECT}' not found on Drive.")
        print("Option B — clone from GitHub instead (edit URL, then uncomment):")
        print("  !git clone https://github.com/YOUR/xai_lime_vs_shap.git /content/xai_lime_vs_shap")
        print("  import os; os.chdir('/content/xai_lime_vs_shap')")

    # Install required packages
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers", "torch", "datasets", "scikit-learn",
         "accelerate", "lime", "shap"],
        check=True,
    )
    print("Packages ready.")
else:
    print("Local environment — no Colab setup needed.")


In [ ]:
## 1. Imports & Config
import os, warnings, json
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


def find_project_root() -> Path:
    """Find repository root robustly across VS Code / Jupyter / Colab."""
    markers = ["data", "notebooks", "src"]
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path("/content/xai_lime_vs_shap"),
    ]

    for candidate in candidates:
        if all((candidate / m).exists() for m in markers):
            return candidate

    # Fallback to current directory if markers are not found
    return Path.cwd()


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "tokopedia_reviews_clean.csv"
MODEL_DIR = PROJECT_ROOT / "outputs" / "indobert_sentiment"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Make sure project folder contains data/processed/tokopedia_reviews_clean.csv"
    )

# Hyperparameters
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

Device: cpu
PyTorch: 2.10.0+cpu


In [3]:
## 2. Load & Prepare Data

df = pd.read_csv(DATA_PATH)

# Label mapping
LABEL2ID = {"Positif": 0, "Negatif": 1, "Netral": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

df["label"] = df["sentiment_label"].map(LABEL2ID)
df = df.dropna(subset=["review_text_clean", "label"])
df["label"] = df["label"].astype(int)

print(f"Total rows: {len(df)}")
print("\nLabel distribution:")
print(df["sentiment_label"].value_counts())

# Train / val / test split  (70/15/15) — stratified
X = df["review_text_clean"].values
y = df["label"].values

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)

print(f"\nTrain: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/tokopedia_reviews_clean.csv'

In [ ]:
## 3. Tokenizer & Dataset class

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = ReviewDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds   = ReviewDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_ds  = ReviewDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print(f"Tokenizer: {MODEL_NAME}")
print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

In [ ]:
## 4. Model & Class Weights

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
model = model.to(DEVICE)

# Compute class weights (for imbalanced data)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=y_train,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print("Class weights:", dict(zip(["Positif","Negatif","Netral"], class_weights.round(3))))

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimizer & scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training steps: {total_steps}")

In [ ]:
## 5. Training Loop

def eval_model(loader, model, loss_fn, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            total_loss += loss.item()
            preds = outputs.logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc = (np.array(all_preds) == np.array(all_labels)).mean()
    return avg_loss, acc, all_preds, all_labels


history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0
    for step, batch in enumerate(train_loader, 1):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        if step % 20 == 0:
            print(f"  Epoch {epoch} step {step}/{len(train_loader)} loss={loss.item():.4f}")

    avg_train = train_loss / len(train_loader)
    val_loss, val_acc, _, _ = eval_model(val_loader, model, loss_fn, DEVICE)
    history["train_loss"].append(avg_train)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    print(f"Epoch {epoch}/{EPOCHS} — train_loss={avg_train:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}\n")

print("Training complete.")

In [ ]:
## 6. Training Curves

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_x = range(1, EPOCHS + 1)

axes[0].plot(epochs_x, history["train_loss"], label="Train")
axes[0].plot(epochs_x, history["val_loss"],   label="Val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(epochs_x, history["val_acc"], color="green", marker="o")
axes[1].set_title("Validation Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].set_ylim(0, 1)

plt.tight_layout()
(PROJECT_ROOT / "outputs" / "figures").mkdir(parents=True, exist_ok=True)
plt.savefig(PROJECT_ROOT / "outputs" / "figures" / "03_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
## 7. Evaluation on Test Set

test_loss, test_acc, test_preds, test_labels = eval_model(test_loader, model, loss_fn, DEVICE)
print(f"Test Loss: {test_loss:.4f}  Test Accuracy: {test_acc:.4f}\n")

target_names = ["Positif", "Negatif", "Netral"]
report = classification_report(test_labels, test_preds, target_names=target_names, zero_division=0)
print(report)

# Save report
(PROJECT_ROOT / "outputs" / "reports").mkdir(parents=True, exist_ok=True)
with open(PROJECT_ROOT / "outputs" / "reports" / "03_classification_report.txt", "w") as f:
    f.write(report)


In [ ]:
## 8. Confusion Matrix

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names)
plt.title("Confusion Matrix — Test Set")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "figures" / "03_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
## 9. Save Model & Tokenizer

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

# Save label map for notebook 04
with open(MODEL_DIR / "label_map.json", "w") as f:
    json.dump({"id2label": ID2LABEL, "label2id": LABEL2ID}, f, indent=2)

print(f"Model saved → {MODEL_DIR}")
print("Contents:", list(MODEL_DIR.glob("*")))